In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "/home/hhoechter/tum/jaxfluids_internship/src")
from compressible_1d import equation_manager

import jax
import jax.numpy as jnp

In [9]:
n = 2000

from compressible_1d import equation_manager_test
import time

t0 = time.perf_counter()
eq_manager = equation_manager_test.create_test_equation_manager()
n_species = eq_manager.species.n_species
n_cells = 10

t1 = time.perf_counter()

U = equation_manager_test.create_test_state(n_cells, n_species)

t2 = time.perf_counter()

@jax.jit
def step_jit(U):
    return equation_manager.advance_one_step(U, eq_manager)

t3 = time.perf_counter()

# First call (compilation)
U1 = step_jit(U)

t4 = time.perf_counter()

for _ in range(n):
    # Second call (use compiled version)
    U2 = step_jit(U)

t44 = time.perf_counter()

U2 = jax.vmap(step_jit, in_axes=(0,))(jnp.broadcast_to(U, (n,) + U.shape))

t43 = time.perf_counter()

def step_fn(U, _):
    U_next = step_jit(U)   # or step(U) and jit outside
    return U_next, None

U2, _ = jax.lax.scan(step_fn, U, xs=None, length=n)

t41 = time.perf_counter()
@jax.jit
def run(U0):
    def body(U, _):
        return step_jit(U), None
    return jax.lax.scan(body, U0, xs=None, length=n)[0]

t42 = time.perf_counter()

U2 = run(U)

t5 = time.perf_counter()


# Results should be identical
assert jnp.allclose(U1, U2), "JIT compilation should not change results"

t6= time.perf_counter()

print(f"Equation manager creation time: {t1 - t0:.6f} seconds")
print(f"Test state creation time: {t2 - t1:.6f} seconds")
print(f"JIT compilation time: {t4 - t3:.6f} seconds")
print(f"JIT execution time (loop): {(t44 - t4):.6f} seconds")
print(f"JIT execution time (vmap): {t43 - t44:.6f} seconds")
print(f"JIT setup time (compilation scan run): {t42 - t41:.6f} seconds")
print(f"JIT execution time scan run: {t5 - t42:.6f} seconds")
print(f"comparison: {t6 - t5:.6f} seconds")
print("advance_one_step JIT compilation test passed")

Equation manager creation time: 0.017485 seconds
Test state creation time: 0.004831 seconds
JIT compilation time: 32.253857 seconds
JIT execution time (loop): 1.036514 seconds
JIT execution time (vmap): 26.642173 seconds
JIT setup time (compilation scan run): 0.000295 seconds
JIT execution time scan run: 22.170150 seconds
comparison: 0.049272 seconds
advance_one_step JIT compilation test passed
